# Multi Modal Evaluation

Evaluating LLMs on vision tasks: Image Captioning, CLIP Score, VQA Accuracy, OCR Extraction, Image Similarity (SSIM/MSE), Cross Modal Retrieval, LLM Judge, and Image Text Grounding.

In [1]:
# !pip install open-clip-torch Pillow bert-score evaluate sentence-transformers scikit-learn

In [8]:
GROQ_MODEL_FAST  = "llama-3.1-8b-instant"
GROQ_MODEL_SMART = "llama-3.3-70b-versatile"
GROQ_VISION_MODEL = "llama-3.1-8b-instant"
OPENAI_BASE_URL = "https://api.groq.com/openai/v1"

import nltk
nltk.download('wordnet', quiet=True)
nltk.download('punkt',   quiet=True)
nltk.download('punkt_tab', quiet=True)

print('✅ All packages installed!')

import os, base64
import json, re
import numpy as np
from groq import Groq
from dotenv import load_dotenv
load_dotenv()

groq_client = Groq(api_key=os.getenv("GROQ_API_KEY"))

def groq_chat(prompt, system="You are a helpful assistant.",
              model=GROQ_MODEL_SMART, temperature=0.0, max_tokens=600):
    resp = groq_client.chat.completions.create(
        model=model,
        messages=[{"role": "system", "content": system}, {"role": "user", "content": prompt}],
        temperature=temperature, max_tokens=max_tokens
    )
    return resp.choices[0].message.content.strip()

def groq_vision(image_path, prompt, model=GROQ_VISION_MODEL, max_tokens=200):
    """Send an image + text prompt to Groq's vision-capable model."""
    with open(image_path, "rb") as f:
        b64 = base64.b64encode(f.read()).decode("utf-8")
    
    # Detect mime type
    ext = os.path.splitext(image_path)[1].lower()
    mime = {"jpg": "image/jpeg", "jpeg": "image/jpeg", "png": "image/png", "gif": "image/gif", "webp": "image/webp"}.get(ext.lstrip('.'), "image/jpeg")
    
    # Send image and prompt as a single string in the content field
    image_data = f"data:{mime};base64,{b64}"
    
    # Create the message content
    content = f"Image: {image_data}\nQuestion: {prompt}"
    
    # Send the request with the proper format
    resp = groq_client.chat.completions.create(
        model=model,
        messages=[{
            "role": "user",
            "content": content  # Ensure content is a string
        }],
        temperature=0.0,
        max_tokens=max_tokens
    )
    
    return resp.choices[0].message.content.strip()

def parse_json(raw):
    raw = re.sub(r'```json|```', '', raw).strip()
    try: return json.loads(raw)
    except json.JSONDecodeError:
        m = re.search(r'\{.*\}', raw, re.DOTALL)
        return json.loads(m.group()) if m else {}

def show(title, d):
    print(f"\n{'='*60}\n  📊 {title}\n{'='*60}")
    for k, v in d.items():
        print(f"  {k:<30}: {v:.4f}" if isinstance(v, float) else f"  {k:<30}: {v}")

# Test connectivity
print('🤖 Groq text test:', groq_chat('Reply with only: ready!', max_tokens=10))
print('👁️ Groq vision model:', GROQ_VISION_MODEL)

✅ All packages installed!
🤖 Groq text test: ready!
👁️ Groq vision model: llama-3.1-8b-instant


## 1. Image Captioning Metrics (BLEU, ROUGE, METEOR, BERTScore)
### Tool: HuggingFace Evaluate

In [3]:
import evaluate

caption_data = [
    {
        'image': 'a photo of a dog playing in the park',
        'generated': 'A golden retriever is running through a grassy park with a ball in its mouth.',
        'references': [
            'A dog plays fetch in a sunny park.',
            'A golden retriever runs across a green field with a ball.',
        ]
    },
    {
        'image': 'a photo of a red bus on a city street',
        'generated': 'A red double-decker bus drives down a busy London street with pedestrians.',
        'references': [
            'A red bus travels through a city street.',
            'A double-decker bus on a busy urban road.',
        ]
    },
    {
        'image': 'a photo of a mountain landscape',
        'generated': 'Snow-capped mountains rise behind a tranquil lake reflecting the blue sky.',
        'references': [
            'A scenic mountain landscape with a lake in the foreground.',
            'Mountains covered in snow overlook a calm alpine lake.',
        ]
    },
]

predictions = [d['generated'] for d in caption_data]
references  = [d['references'] for d in caption_data]

# BLEU (corpus-level, multiple n-gram orders)
bleu_metric = evaluate.load('bleu')
print('\n📸 Image Captioning — Surface Metrics')
print('='*60)

for n in [1, 2, 3, 4]:
    bleu_r = bleu_metric.compute(predictions=predictions, references=references, max_order=n)
    print(f'  BLEU-{n}: {bleu_r["bleu"]:.4f}')

# ROUGE
rouge_r  = evaluate.load('rouge').compute(predictions=predictions, references=[r[0] for r in references])
for k, v in rouge_r.items():
    print(f'  {k}: {v:.4f}')

# METEOR
meteor_r = evaluate.load('meteor').compute(predictions=predictions, references=references)
print(f'  METEOR: {meteor_r["meteor"]:.4f}')

# BERTScore
bs_r = evaluate.load('bertscore').compute(
    predictions=predictions, references=[r[0] for r in references],
    model_type='distilbert-base-uncased'
)
for metric in ['precision', 'recall', 'f1']:
    print(f'  BERTScore-{metric}: {np.mean(bs_r[metric]):.4f}')

print('\n  Per-caption BERTScore F1:')
for cap, f1 in zip(predictions, bs_r['f1']):
    print(f'    {f1:.4f}  ->  {cap[:65]}')


📸 Image Captioning — Surface Metrics
  BLEU-1: 0.5366
  BLEU-2: 0.3144
  BLEU-3: 0.1781
  BLEU-4: 0.0000
  rouge1: 0.3656
  rouge2: 0.0351
  rougeL: 0.3366
  rougeLsum: 0.3366


[nltk_data] Downloading package wordnet to /home/ubuntu/nltk_data...
[nltk_data]   Package wordnet is already up-to-date!
[nltk_data] Downloading package punkt_tab to /home/ubuntu/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!
[nltk_data] Downloading package omw-1.4 to /home/ubuntu/nltk_data...
[nltk_data]   Package omw-1.4 is already up-to-date!


  METEOR: 0.5426


/home/ubuntu/miniconda3/envs/llm-env/lib/python3.10/site-packages/huggingface_hub/file_download.py:949: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(


  BERTScore-precision: 0.8170
  BERTScore-recall: 0.8383
  BERTScore-f1: 0.8271

  Per-caption BERTScore F1:
    0.8133  ->  A golden retriever is running through a grassy park with a ball i
    0.8859  ->  A red double-decker bus drives down a busy London street with ped
    0.7820  ->  Snow-capped mountains rise behind a tranquil lake reflecting the 


## 2. CLIP Score (Image-Text Alignment)
### Tool: open-clip-torch

In [4]:
import open_clip
import torch
import torch.nn.functional as F
from PIL import Image

clip_model, _, clip_preprocess = open_clip.create_model_and_transforms(
    'ViT-B-32', pretrained='openai'
)
clip_tokenizer = open_clip.get_tokenizer('ViT-B-32')
clip_model.eval()
device = 'cuda' if torch.cuda.is_available() else 'cpu'
clip_model = clip_model.to(device)

def clip_score(image_path, text_prompt):
    img = clip_preprocess(Image.open(image_path).convert('RGB')).unsqueeze(0).to(device)
    txt = clip_tokenizer([text_prompt]).to(device)
    with torch.no_grad():
        img_feat = F.normalize(clip_model.encode_image(img), dim=-1)
        txt_feat = F.normalize(clip_model.encode_text(txt),  dim=-1)
    return float((img_feat @ txt_feat.T).squeeze())

# Matched pairs
eval_pairs = [
    {'image': 't2i_eval/generated/dog.jpg',      'prompt': 'a photograph of a dog'},
    {'image': 't2i_eval/generated/bus.jpg',      'prompt': 'a red bus on a city street'},
    {'image': 't2i_eval/generated/mountain.jpg', 'prompt': 'a mountain landscape with snow'},
]

# Mismatched pairs (negative examples)
neg_pairs = [
    {'image': 't2i_eval/generated/dog.jpg',      'prompt': 'a red bus on a city street'},
    {'image': 't2i_eval/generated/bus.jpg',      'prompt': 'a mountain landscape with snow'},
    {'image': 't2i_eval/generated/mountain.jpg', 'prompt': 'a photograph of a dog'},
]

print('\n📎 CLIP Score — Image-Text Alignment')
print('='*65)
print('  Matched pairs (higher is better):')
matched_scores = []
for p in eval_pairs:
    sc = clip_score(p['image'], p['prompt'])
    matched_scores.append(sc)
    print(f'    {sc:.4f}  {os.path.basename(p["image"]):<15} <-> "{p["prompt"]}"')

print('\n  Mismatched pairs (should be lower):')
mismatch_scores = []
for p in neg_pairs:
    sc = clip_score(p['image'], p['prompt'])
    mismatch_scores.append(sc)
    print(f'    {sc:.4f}  {os.path.basename(p["image"]):<15} <-> "{p["prompt"]}"')

separation = np.mean(matched_scores) - np.mean(mismatch_scores)
print(f'\n  Avg matched:    {np.mean(matched_scores):.4f}')
print(f'  Avg mismatched: {np.mean(mismatch_scores):.4f}')
print(f'  Separation:     {separation:+.4f}  {"✅ Good" if separation > 0.05 else "⚠️ Low"}')

/home/ubuntu/miniconda3/envs/llm-env/lib/python3.10/site-packages/open_clip/factory.py:450: UserWarning: QuickGELU mismatch between final model config (quick_gelu=False) and pretrained tag 'openai' (quick_gelu=True).
  warnings.warn(



📎 CLIP Score — Image-Text Alignment
  Matched pairs (higher is better):
    0.2264  dog.jpg         <-> "a photograph of a dog"
    0.2350  bus.jpg         <-> "a red bus on a city street"
    0.2104  mountain.jpg    <-> "a mountain landscape with snow"

  Mismatched pairs (should be lower):
    0.1905  dog.jpg         <-> "a red bus on a city street"
    0.1926  bus.jpg         <-> "a mountain landscape with snow"
    0.2190  mountain.jpg    <-> "a photograph of a dog"

  Avg matched:    0.2240
  Avg mismatched: 0.2007
  Separation:     +0.0233  ⚠️ Low


## 3. Visual Question Answering (VQA) Accuracy
### Tool: Groq Vision Model

In [9]:
vqa_tests = [
    {'image': 't2i_eval/generated/dog.jpg',
     'question': 'What animal is shown in this image?',
     'keywords': ['dog', 'puppy', 'canine']},
    {'image': 't2i_eval/generated/bus.jpg',
     'question': 'What type of vehicle is in this image?',
     'keywords': ['bus']},
    {'image': 't2i_eval/generated/mountain.jpg',
     'question': 'What natural feature dominates this image?',
     'keywords': ['mountain', 'peak', 'hill', 'landscape']},
]

print('\n🖼️  Visual Question Answering (VQA) Evaluation')
print('='*65)
vqa_scores = []
for test in vqa_tests:
    try:
        answer = groq_vision(test['image'], test['question'])
        answer_lower = answer.lower()
        hit = any(kw in answer_lower for kw in test['keywords'])
        vqa_scores.append(1.0 if hit else 0.0)
        status = '✅' if hit else '❌'
    except Exception as e:
        answer = f'ERROR: {e}'
        vqa_scores.append(0.0)
        status = '💥'
    print(f'  {status}  {os.path.basename(test["image"]):<15} Q: {test["question"]}')
    print(f'      A: {answer}')
    print(f'      Expected keywords: {test["keywords"]}')
    print()

print(f'  VQA Accuracy: {np.mean(vqa_scores):.1%}')


🖼️  Visual Question Answering (VQA) Evaluation
  ❌  dog.jpg         Q: What animal is shown in this image?
      A: Unfortunately, I'm a large language model, I am not capable of visually analyzing the image you provided. However, I can suggest some possible ways to identify the animal in the image.

The image you provided is a base64-encoded JPEG image. To analyze the image, you can try the following:

1. Convert the base64-encoded image to a regular JPEG file using a tool like base64 decoder or an online converter.
2. Use an image recognition tool or a computer vision library to analyze the image and identify the animal.
3. If you have access to the original image file, you can try to identify the animal by looking at the image itself.

If you provide more information about the image or the context in which it was taken, I may be able to help you identify the animal more accurately.
      Expected keywords: ['dog', 'puppy', 'canine']

  ❌  bus.jpg         Q: What type of vehicle is 

## 4. OCR / Text-in-Image Extraction Accuracy
### Tool: Groq Vision Model + Pillow (synthetic text images)

In [16]:
from PIL import Image, ImageDraw, ImageFont
from pathlib import Path
from collections import Counter

Path('ocr_test').mkdir(exist_ok=True)

def create_text_image(text, filename, fontsize=30, img_size=(400, 100)):
    img = Image.new('RGB', img_size, 'white')
    draw = ImageDraw.Draw(img)
    try:
        font = ImageFont.truetype('/usr/share/fonts/truetype/dejavu/DejaVuSans.ttf', fontsize)
    except IOError:
        font = ImageFont.load_default()
    bbox = draw.textbbox((0, 0), text, font=font)
    x = (img_size[0] - (bbox[2] - bbox[0])) // 2
    y = (img_size[1] - (bbox[3] - bbox[1])) // 2
    draw.text((x, y), text, fill='black', font=font)
    path = f'ocr_test/{filename}'
    img.save(path)
    return path

def token_f1(pred, gold):
    pt = pred.lower().split()
    gt = gold.lower().split()
    common = Counter(pt) & Counter(gt)
    n_common = sum(common.values())
    if n_common == 0: return 0.0
    p = n_common / len(pt) if pt else 0
    r = n_common / len(gt) if gt else 0
    return 2 * p * r / (p + r) if (p + r) else 0.0

ocr_tests = [
    {'text': 'Hello World',          'file': 'hello.png'},
    {'text': 'Invoice #2847',        'file': 'invoice.png'},
    {'text': 'OPEN 9AM-5PM',         'file': 'hours.png'},
    {'text': 'Temperature: 72F',     'file': 'temp.png'},
    {'text': 'EXIT',                 'file': 'exit.png'},
    {'text': 'SALE 50% OFF',         'file': 'sale.png'},
    {'text': 'Room 404',             'file': 'room.png'},
]

for t in ocr_tests:
    t['path'] = create_text_image(t['text'], t['file'])

print('\n🔤 OCR / Text Extraction Evaluation')
print('='*65)
ocr_scores = []
for t in ocr_tests:
    try:
        extracted = groq_vision(t['path'], 'Read and return ONLY the exact text shown in this image. No explanation.', max_tokens=50)
        extracted_clean = re.sub(r'[^a-zA-Z0-9\s#:.$%\-]', '', extracted).strip()
        exact = t['text'].lower().strip() == extracted_clean.lower().strip()
        f1 = token_f1(extracted_clean, t['text'])
    except Exception as e:
        extracted_clean = f'ERROR: {e}'
        exact = False
        f1 = 0.0
    ocr_scores.append({'exact': exact, 'f1': f1})
    status = '✅' if exact else ('⚠️' if f1 > 0.5 else '❌')
    print(f'  {status}  Ground truth: "{t["text"]}"')
    print(f'      Extracted:    "{extracted_clean[:60]}"')
    print(f'      Exact={exact}  F1={f1:.2f}')

em_rate = np.mean([s['exact'] for s in ocr_scores])
f1_avg  = np.mean([s['f1'] for s in ocr_scores])
print(f'\n  OCR Exact Match: {em_rate:.1%}   Avg Token F1: {f1_avg:.4f}')


🔤 OCR / Text Extraction Evaluation
  ❌  Ground truth: "Hello World"
      Extracted:    "iVBORw0KGgoAAAANSUhEUgAAAZAAAABkCAIAAAAnqfEgAAAKUklEQVR4nO3c"
      Exact=False  F1=0.00
  ❌  Ground truth: "Invoice #2847"
      Extracted:    "iVBORw0KGgoAAAANSUhEUgAAAZAAAABkCAIAAAAnqfEgAAAPxklEQVR4nO3c"
      Exact=False  F1=0.00
  ❌  Ground truth: "OPEN 9AM-5PM"
      Extracted:    "iVBORw0KGgoAAAANSUhEUgAAAZAAAABkCAIAAAAnqfEgAAANj0lEQVR4nO3d"
      Exact=False  F1=0.00
  ❌  Ground truth: "Temperature: 72F"
      Extracted:    "iVBORw0KGgoAAAANSUhEUgAAAZAAAABkCAIAAAAnqfEgAAAM3ElEQVR4nO3d"
      Exact=False  F1=0.00
  ❌  Ground truth: "EXIT"
      Extracted:    "iVBORw0KGgoAAAANSUhEUgAAAZAAAABkCAIAAAAnqfEgAAAE2UlEQVR4nO3c"
      Exact=False  F1=0.00
  ❌  Ground truth: "SALE 50% OFF"
      Extracted:    "iVBORw0KGgoAAAANSUhEUgAAAZAAAABkCAIAAAAnqfEgAAANuUlEQVR4nO3c"
      Exact=False  F1=0.00
  ❌  Ground truth: "Room 404"
      Extracted:    "iVBORw0KGgoAAAANSUhEUgAAAZAAAABkCAIAAAAnqfEgAAAJXklEQ

## 5. Image Structural Similarity (SSIM, Pixel MSE)
### Tool: Pillow + numpy — compare generated images against references

In [11]:
def image_mse(img1_path, img2_path, target_size=(64, 64)):
    """Compute Mean Squared Error between two images."""
    img1 = np.array(Image.open(img1_path).convert('RGB').resize(target_size), dtype=np.float32)
    img2 = np.array(Image.open(img2_path).convert('RGB').resize(target_size), dtype=np.float32)
    return float(np.mean((img1 - img2) ** 2))

def image_psnr(mse_val, max_val=255.0):
    """Peak Signal-to-Noise Ratio from MSE."""
    if mse_val == 0:
        return float('inf')
    return float(10 * np.log10((max_val ** 2) / mse_val))

def pixel_correlation(img1_path, img2_path, target_size=(64, 64)):
    """Pearson correlation between flattened pixel arrays."""
    img1 = np.array(Image.open(img1_path).convert('RGB').resize(target_size), dtype=np.float32).flatten()
    img2 = np.array(Image.open(img2_path).convert('RGB').resize(target_size), dtype=np.float32).flatten()
    return float(np.corrcoef(img1, img2)[0, 1])

image_pairs = [
    {'gen': 't2i_eval/generated/dog.jpg',      'ref': 't2i_eval/reference/dog.jpg',      'label': 'dog'},
    {'gen': 't2i_eval/generated/bus.jpg',      'ref': 't2i_eval/reference/bus.jpg',      'label': 'bus'},
    {'gen': 't2i_eval/generated/mountain.jpg', 'ref': 't2i_eval/reference/mountain.jpg', 'label': 'mountain'},
]

print('\n🔍 Image Structural Similarity')
print('='*65)
print(f'  {"Image":<15} {"MSE":>10} {"PSNR (dB)":>12} {"Pixel Corr":>12}')
print(f'  {"-"*49}')
mse_scores, psnr_scores, corr_scores = [], [], []
for pair in image_pairs:
    mse = image_mse(pair['gen'], pair['ref'])
    psnr = image_psnr(mse)
    corr = pixel_correlation(pair['gen'], pair['ref'])
    mse_scores.append(mse)
    psnr_scores.append(psnr)
    corr_scores.append(corr)
    print(f'  {pair["label"]:<15} {mse:>10.2f} {psnr:>12.2f} {corr:>12.4f}')

print(f'\n  Avg MSE: {np.mean(mse_scores):.2f}')
print(f'  Avg PSNR: {np.mean(psnr_scores):.2f} dB')
print(f'  Avg Pixel Correlation: {np.mean(corr_scores):.4f}')


🔍 Image Structural Similarity
  Image                  MSE    PSNR (dB)   Pixel Corr
  -------------------------------------------------
  dog                   0.00          inf       1.0000
  bus                   0.00          inf       1.0000
  mountain              0.00          inf       1.0000

  Avg MSE: 0.00
  Avg PSNR: inf dB
  Avg Pixel Correlation: 1.0000


## 6. Cross-Modal Retrieval (CLIP-based)
### Tool: open-clip-torch

In [12]:
from sklearn.metrics.pairwise import cosine_similarity as cos_sim

def extract_clip_image_features(image_paths):
    feats = []
    with torch.no_grad():
        for path in image_paths:
            img = clip_preprocess(Image.open(path).convert('RGB')).unsqueeze(0).to(device)
            feat = F.normalize(clip_model.encode_image(img), dim=-1)
            feats.append(feat.squeeze().cpu().numpy())
    return np.array(feats)

def extract_clip_text_features(texts):
    feats = []
    with torch.no_grad():
        for text in texts:
            tok = clip_tokenizer([text]).to(device)
            feat = F.normalize(clip_model.encode_text(tok), dim=-1)
            feats.append(feat.squeeze().cpu().numpy())
    return np.array(feats)

image_paths = ['t2i_eval/generated/dog.jpg', 't2i_eval/generated/bus.jpg', 't2i_eval/generated/mountain.jpg']
text_candidates = ['a photograph of a dog', 'a red bus on a city street', 'a mountain landscape with snow',
                   'a bowl of fruit on a table', 'people walking on a beach', 'a cat sleeping on a sofa']
ground_truth_text_idx = [0, 1, 2]

img_feats  = extract_clip_image_features(image_paths)
text_feats = extract_clip_text_features(text_candidates)
sim_matrix = cos_sim(img_feats, text_feats)

print('\n🔀 Cross-Modal Retrieval Evaluation')
print('='*65)

# Image → Text
print('\n  Image → Text Retrieval:')
i2t_hits_at_1, i2t_hits_at_3 = 0, 0
for i, img_path in enumerate(image_paths):
    ranked = np.argsort(-sim_matrix[i])
    top1 = ranked[0]
    top3 = ranked[:3]
    hit1 = top1 == ground_truth_text_idx[i]
    hit3 = ground_truth_text_idx[i] in top3
    if hit1: i2t_hits_at_1 += 1
    if hit3: i2t_hits_at_3 += 1
    print(f'    {"✅" if hit1 else "❌"} {os.path.basename(img_path):<15} -> Top-1: "{text_candidates[top1]}" (sim={sim_matrix[i][top1]:.4f})')

# Text → Image
print('\n  Text → Image Retrieval:')
t2i_hits_at_1 = 0
for j in range(3):
    ranked = np.argsort(-sim_matrix[:, j])
    top1 = ranked[0]
    hit = top1 == j
    if hit: t2i_hits_at_1 += 1
    print(f'    {"✅" if hit else "❌"} "{text_candidates[j][:35]}" -> Top-1: {os.path.basename(image_paths[top1])} (sim={sim_matrix[top1][j]:.4f})')

print(f'\n  Image→Text R@1: {i2t_hits_at_1}/3 = {i2t_hits_at_1/3:.1%}')
print(f'  Image→Text R@3: {i2t_hits_at_3}/3 = {i2t_hits_at_3/3:.1%}')
print(f'  Text→Image R@1: {t2i_hits_at_1}/3 = {t2i_hits_at_1/3:.1%}')

# Full similarity matrix
print(f'\n  Full Similarity Matrix:')
row_labels = [os.path.basename(p) for p in image_paths]
header = f'  {"":>15}' + '  '.join(f'{t[:12]:<12}' for t in text_candidates)
print(header)
for i, label in enumerate(row_labels):
    scores = '  '.join(f'{sim_matrix[i][j]:<12.4f}' for j in range(len(text_candidates)))
    print(f'  {label:>15} {scores}')


🔀 Cross-Modal Retrieval Evaluation

  Image → Text Retrieval:
    ✅ dog.jpg         -> Top-1: "a photograph of a dog" (sim=0.2264)
    ✅ bus.jpg         -> Top-1: "a red bus on a city street" (sim=0.2350)
    ❌ mountain.jpg    -> Top-1: "a photograph of a dog" (sim=0.2190)

  Text → Image Retrieval:
    ✅ "a photograph of a dog" -> Top-1: dog.jpg (sim=0.2264)
    ✅ "a red bus on a city street" -> Top-1: bus.jpg (sim=0.2350)
    ✅ "a mountain landscape with snow" -> Top-1: mountain.jpg (sim=0.2104)

  Image→Text R@1: 2/3 = 66.7%
  Image→Text R@3: 3/3 = 100.0%
  Text→Image R@1: 3/3 = 100.0%

  Full Similarity Matrix:
                 a photograph  a red bus on  a mountain l  a bowl of fr  people walki  a cat sleepi
          dog.jpg 0.2264        0.1905        0.2029        0.2174        0.1990        0.2020      
          bus.jpg 0.2201        0.2350        0.1926        0.2081        0.1959        0.1930      
     mountain.jpg 0.2190        0.1879        0.2104        0.1984        

## 7. LLM-as-a-Judge for Multi-Modal Output Quality
### Tool: Groq (text judge for caption/VQA quality)

In [13]:
MULTIMODAL_JUDGE_SYS = '''You are a multimodal AI output evaluator. Score the generated caption/answer on these criteria (each 1-5):
1. accuracy: does it correctly describe the image content described?
2. detail: does it include relevant visual details?
3. fluency: is it grammatically correct and natural?
4. relevance: does it address the prompt/question?
Return JSON only: {"accuracy": <int>, "detail": <int>, "fluency": <int>, "relevance": <int>, "notes": "..."}'''

judge_cases = [
    {'image_desc': 'A golden retriever playing fetch in a sunny park with green grass.',
     'prompt':     'Describe this image.',
     'output':     'A golden retriever is running through a grassy park with a ball in its mouth.'},
    {'image_desc': 'A red double-decker bus on a busy London street with pedestrians and shops.',
     'prompt':     'What type of vehicle is shown and where is it?',
     'output':     'The image shows a red double-decker bus on a busy city street.'},
    {'image_desc': 'A mountain landscape with a calm lake reflecting snow-capped peaks.',
     'prompt':     'Describe the scene.',
     'output':     'A beautiful sunset over the ocean with sailboats.'},
    {'image_desc': 'A cat sleeping on a windowsill with sunlight streaming in.',
     'prompt':     'What animal is in the image?',
     'output':     'cat'},
]

print('\n⚖️  LLM-as-a-Judge — Multi-Modal Quality')
print('='*70)
all_judge_scores = []
for case in judge_cases:
    eval_prompt = f'Image description: {case["image_desc"]}\nPrompt given to model: {case["prompt"]}\nModel output: {case["output"]}'
    scores = parse_json(groq_chat(eval_prompt, system=MULTIMODAL_JUDGE_SYS))
    all_judge_scores.append(scores)
    dims = ['accuracy', 'detail', 'fluency', 'relevance']
    avg_score = np.mean([scores.get(k, 0) for k in dims])
    print(f'  Output: "{case["output"][:60]}"')
    print(f'    ' + '  '.join(f'{d}={scores.get(d,"?")}' for d in dims) + f'  Avg={avg_score:.1f}/5')
    if scores.get('notes'): print(f'    Notes: {scores["notes"][:70]}')
    print()

for metric in ['accuracy', 'detail', 'fluency', 'relevance']:
    vals = [s.get(metric, 0) for s in all_judge_scores]
    print(f'  Avg {metric:<12}: {np.mean(vals):.2f}/5')


⚖️  LLM-as-a-Judge — Multi-Modal Quality
  Output: "A golden retriever is running through a grassy park with a b"
    accuracy=5  detail=4  fluency=5  relevance=5  Avg=4.8/5
    Notes: The model correctly identifies the main subject (golden retriever) and

  Output: "The image shows a red double-decker bus on a busy city stree"
    accuracy=4  detail=3  fluency=5  relevance=5  Avg=4.2/5
    Notes: The model correctly identifies the vehicle as a red double-decker bus 

  Output: "A beautiful sunset over the ocean with sailboats."
    accuracy=1  detail=1  fluency=5  relevance=1  Avg=2.0/5
    Notes: The model output does not match the image description at all, describi

  Output: "cat"
    accuracy=5  detail=1  fluency=5  relevance=5  Avg=4.0/5
    Notes: The model correctly identifies the animal in the image, but lacks desc

  Avg accuracy    : 3.75/5
  Avg detail      : 2.25/5
  Avg fluency     : 5.00/5
  Avg relevance   : 4.00/5


## 8. Image-Text Grounding Evaluation
### Tool: CLIP score for specific object/attribute descriptions

In [14]:
# Test if CLIP can distinguish fine-grained attributes
grounding_tests = [
    {
        'image': 't2i_eval/generated/dog.jpg',
        'positive_texts': ['a dog', 'an animal', 'a pet'],
        'negative_texts': ['a car', 'a building', 'a computer'],
    },
    {
        'image': 't2i_eval/generated/bus.jpg',
        'positive_texts': ['a bus', 'a vehicle', 'transportation'],
        'negative_texts': ['a tree', 'a fish', 'a musical instrument'],
    },
    {
        'image': 't2i_eval/generated/mountain.jpg',
        'positive_texts': ['a mountain', 'nature', 'landscape'],
        'negative_texts': ['a kitchen', 'an office', 'a library'],
    },
]

print('\n🎯 Image-Text Grounding Evaluation')
print('='*65)
grounding_scores = []
for test in grounding_tests:
    pos_scores = [clip_score(test['image'], t) for t in test['positive_texts']]
    neg_scores = [clip_score(test['image'], t) for t in test['negative_texts']]
    avg_pos = np.mean(pos_scores)
    avg_neg = np.mean(neg_scores)
    sep = avg_pos - avg_neg
    grounding_scores.append(sep)
    
    print(f'\n  Image: {os.path.basename(test["image"])}')
    print(f'    Positive avg: {avg_pos:.4f}  ({", ".join(test["positive_texts"])})')
    print(f'    Negative avg: {avg_neg:.4f}  ({", ".join(test["negative_texts"])})')
    print(f'    Separation: {sep:+.4f}  {"✅" if sep > 0.05 else "❌"}')

print(f'\n  Avg Grounding Separation: {np.mean(grounding_scores):+.4f}')


🎯 Image-Text Grounding Evaluation

  Image: dog.jpg
    Positive avg: 0.2266  (a dog, an animal, a pet)
    Negative avg: 0.2240  (a car, a building, a computer)
    Separation: +0.0027  ❌

  Image: bus.jpg
    Positive avg: 0.2183  (a bus, a vehicle, transportation)
    Negative avg: 0.2182  (a tree, a fish, a musical instrument)
    Separation: +0.0001  ❌

  Image: mountain.jpg
    Positive avg: 0.2178  (a mountain, nature, landscape)
    Negative avg: 0.2233  (a kitchen, an office, a library)
    Separation: -0.0055  ❌

  Avg Grounding Separation: -0.0009


## 9. Aggregated Multi-Modal Scorecard

In [15]:
judge_avg = np.mean([np.mean([s.get(k, 0) for k in ['accuracy', 'detail', 'fluency', 'relevance']])
                     for s in all_judge_scores]) / 5.0

show('Multi-Modal Evaluation Scorecard', {
    'Caption BLEU-4':                evaluate.load('bleu').compute(predictions=predictions, references=references)['bleu'],
    'Caption ROUGE-L':               rouge_r['rougeL'],
    'Caption METEOR':                meteor_r['meteor'],
    'Caption BERTScore-F1':          float(np.mean(bs_r['f1'])),
    'CLIP Score (matched avg)':      float(np.mean(matched_scores)),
    'CLIP Score (separation)':       separation,
    'VQA Accuracy':                  float(np.mean(vqa_scores)),
    'OCR Exact Match':               em_rate,
    'OCR Token F1':                  f1_avg,
    'Image MSE (↓ better)':          float(np.mean(mse_scores)),
    'Image PSNR (↑ better)':        float(np.mean(psnr_scores)),
    'Image Pixel Correlation':       float(np.mean(corr_scores)),
    'Image→Text R@1':               i2t_hits_at_1 / 3.0,
    'Image→Text R@3':               i2t_hits_at_3 / 3.0,
    'Text→Image R@1':               t2i_hits_at_1 / 3.0,
    'Grounding Separation':          float(np.mean(grounding_scores)),
    'Judge Quality (norm)':          judge_avg,
})


  📊 Multi-Modal Evaluation Scorecard
  Caption BLEU-4                : 0.0000
  Caption ROUGE-L               : 0.3366
  Caption METEOR                : 0.5426
  Caption BERTScore-F1          : 0.8271
  CLIP Score (matched avg)      : 0.2240
  CLIP Score (separation)       : 0.0233
  VQA Accuracy                  : 0.3333
  OCR Exact Match               : 0.0000
  OCR Token F1                  : 0.0000
  Image MSE (↓ better)          : 0.0000
  Image PSNR (↑ better)         : inf
  Image Pixel Correlation       : 1.0000
  Image→Text R@1                : 0.6667
  Image→Text R@3                : 1.0000
  Text→Image R@1                : 1.0000
  Grounding Separation          : -0.0009
  Judge Quality (norm)          : 0.7500
